# UABR-SLM · Notebook 08 — Phase 6: SFT + replay arms E2–E7 (validation; test stays sealed for notebook 09)

**Question:** at an equal replay-token budget, does boundary × utility priority (UABR) preserve the pre-SFT routing
policy better than random, Hybrid-CASR-adapted, OPR-adapted, boundary-only and benefit-only replay? GPU **T4** (×1) ·
Internet **On** not needed · ~3–3.5 h (16 SFT runs of ~7.5 min + evaluation). Safe to re-run with this notebook's previous
output attached: finished runs are reused.

**Fixed design (team decisions, 11 Nov 2026, before any Phase 6 result)**

| Item | Value |
|---|---|
| Start | θ_route,preSFT (`06_routing/router_action.pt`) = E0 |
| SFT | identical to notebook 07 (same 34,884 examples / 6,000,100 tokens, same batches per seed, 1,091 updates, LR 5e-5 const, batch 32); seeds 11, 29 |
| E1 | notebook 07's 100 % checkpoints (SFT only) |
| Replay example | **action-token only**: `<BOS><USER> q<ASSISTANT>`, 2-logit CE over {`<ANSWER>`, `<CALL>`} (router loss); target = frozen θ0 label y |
| Loss | L = L_SFT + β · L_replay; **β tuned on validation with the method-neutral random arm (E2, seed 11)** from {0.1, 0.3, 1, 3}: smallest β within 0.01 AUROC of the best; then fixed for every arm |
| Pool | half-B questions the router was trained on, scored once with θ_route,preSFT (m = \|z_CALL − z_ANSWER\|, τ = median m) |
| Budget | each arm selects distinct examples worth **25 % of the pool's tokens**, cycled to **5 % of the instruction tokens**; replay tokens are spread evenly over the 1,091 updates |
| Class ratio | **identical for every arm** = the pool's ANSWER:CALL token share; methods rank within each class |
| Arms | E2 random · E3 Hybrid-CASR-adapted (θ0 answer uncertainty = first-token entropy) · E4 OPR-adapted (current model re-scores the pool **every 100 updates**; prompts whose current decision equals the frozen label are kept; random selection at the same budget/ratio) · E5 boundary-only (smallest m) · E6 benefit-only (largest \|b_net\|) · E7 UABR (exp(−m/τ)·\|b_net\|) |
| Primary endpoint | **routing AUROC vs the frozen θ0 labels** (A3a) + decision agreement with E0; fresh-reward utility (EM and answer-containment) is secondary (A3b) |
| Here | **validation only, preliminary**; notebook 09 opens the sealed test split once for every arm with the claim rule fixed in advance |

**Inputs:** outputs of notebooks **00, 01, 02, 06 (v2), 07** (+ optionally a previous run of this notebook).

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em, alias_contains
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, auprc, clustered_bootstrap_auroc
from uabr.routing.pilot import decision_slot, top_fraction, outcome, paired_cluster_bootstrap
from uabr.routing.router import router_prompt
from uabr.replay.selectors import ARMS, STATIC, priority, class_budgets, select_by_class, Cycler, ReplaySchedule, jaccard
from uabr.train.sft import STANDARD, instruction_batches, train_sft, sft_val_loss

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/08_replay" if ON_KAGGLE else os.path.abspath("outputs/08_replay")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"

SEEDS = [int(s) for s in os.environ.get("SEEDS", "11,29").split(",")]
BETA_GRID = [float(b) for b in os.environ.get("BETA_GRID", "0.1,0.3,1,3").split(",")]
BETA_TOL = 0.01
SELECT_FRAC, REPLAY_FRAC = 0.25, 0.05
OPR_EVERY = int(os.environ.get("OPR_EVERY", 100))
CFG = dict(STANDARD)
N_BOOT = int(os.environ.get("N_BOOT", 1000))
BUDGETS = [0.05, 0.10, 0.20]
MAX_NEW, SEQ = 16, 512
MAX_HOURS = float(os.environ.get("MAX_HOURS", 11.0))          # stop starting new runs after this (Kaggle 12 h limit)
LIMIT = int(os.environ.get("LIMIT", 0))

def one_input(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]

tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
POLICY_PATH = one_input("06_routing/routing_policy.json", "POLICY")
POLICY = json.load(open(POLICY_PATH)); D06 = os.path.dirname(POLICY_PATH)
assert POLICY["format"] == "action"
ROUTER = os.path.join(D06, "router_action.pt"); LAM = float(POLICY["lambda"])
REPORT07_PATH = one_input("07_sft_pilot/pilot_report.json", "REPORT07"); D07 = os.path.dirname(REPORT07_PATH)
REPORT07 = json.load(open(REPORT07_PATH))
assert REPORT07["decision"] == "GO", REPORT07["decision"]
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet"))
B = pd.read_parquet(os.path.join(D06, "labels_halfB.parquet"))
for c in ["y", "b_net", "first_entropy", "question", "group_id"]:
    assert c in B.columns, f"labels_halfB.parquet has no column {c} - attach the 06 v2 output"
if LIMIT: V, B = V.head(LIMIT), B.head(4 * LIMIT)
V, B = V.reset_index(drop=True), B.reset_index(drop=True)
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]

# SFT data exactly as notebook 07
sx = pd.read_parquet(os.path.join(D07, "sft_examples.parquet"))
mk = lambda r: (list(map(int, r.tokens)), [False] * int(r.n_prompt) + [True] * (len(r.tokens) - int(r.n_prompt)))
SFT_TRAIN = [mk(r) for r in sx[sx.part == "train"].itertuples(index=False)]
SFT_VALX = [mk(r) for r in sx[sx.part == "val"].itertuples(index=False)]
INSTR_TOKENS = sum(len(t) for t, _ in SFT_TRAIN)
assert INSTR_TOKENS == REPORT07["sft_data"]["train_tokens"] and len(SFT_TRAIN) == REPORT07["sft_data"]["train_examples"]
E1 = {s: os.path.join(D07, f"sft_s{s}", "final.pt") for s in SEEDS}
for s, p in E1.items(): assert os.path.exists(p), f"07 checkpoint missing: {p}"
print(f"theta_route = {ROUTER} | lambda {LAM} | val {len(V):,} | half-B pool {len(B):,} | SFT {len(SFT_TRAIN):,} examples, {INSTR_TOKENS:,} tokens | E1 from {D07}")

## 3 · Replay pool (half B, scored once with θ_route,preSFT), budgets and selections

In [ ]:
m0 = load_model(ROUTER, DEV)
POOL = pd.DataFrame({"id": B.id.values, "y": B.y.astype(int).values, "b_net": B.b_net.values, "unc": B.first_entropy.values,
                     "group_id": B.group_id.values, "prompt": [router_prompt(tok, q, "action") for q in B.question]})
POOL["tokens"] = POOL.prompt.map(len)
d0 = decision_slot(m0, POOL.prompt.tolist(), DEV); del m0
POOL["score0"], POOL["margin"] = d0.score.values, d0.margin.values
TAU = float(POOL.margin.median())
POOL_TOKENS = int(POOL.tokens.sum()); SELECT_TOKENS = int(SELECT_FRAC * POOL_TOKENS); REPLAY_TOKENS = int(REPLAY_FRAC * INSTR_TOKENS)
BUD = class_budgets(POOL, SELECT_TOKENS)
print(f"pool {len(POOL):,} prompts / {POOL_TOKENS:,} tokens | CALL labels {POOL.y.mean():.1%} (token share {POOL.loc[POOL.y==1,'tokens'].sum()/POOL_TOKENS:.1%}) "
      f"| tau (median m) {TAU:.3f} | select {SELECT_TOKENS:,} tokens per arm {BUD} | replay {REPLAY_TOKENS:,} tokens per run")
print(f"pool router AUROC vs labels (training data): {auroc(POOL.y, POOL.score0):.3f} | default call rate {np.mean(POOL.score0 > 0):.1%}")
SEL = {}
for seed in SEEDS:
    for meth in STATIC:
        SEL[(meth, seed)] = select_by_class(POOL, priority(POOL, meth, seed=seed, tau=TAU), BUD)
rows = []
for meth in STATIC:
    s = SEL[(meth, SEEDS[0])]
    rows.append({"method": meth, "distinct": len(s), "tokens": int(s.tokens.sum()), "answer/call": f"{(s.y==0).sum()}/{(s.y==1).sum()}",
                 "mean_m": s.margin.mean(), "mean_|b_net|": s.b_net.abs().mean(), "mean_unc": s.unc.mean(),
                 "share_b_net>0": (s.b_net > 0).mean()})
SEL_REPORT = pd.DataFrame(rows); display(SEL_REPORT.round(3))
JAC = pd.DataFrame([[jaccard(SEL[(a, SEEDS[0])].id, SEL[(b, SEEDS[0])].id) for b in STATIC] for a in STATIC], index=STATIC, columns=STATIC)
print("Jaccard overlap of selected prompts (seed", SEEDS[0], ")"); display(JAC.round(3))

## 4 · Replay schedules and the run helper

In [ ]:
def make_replay(method, seed, total):
    sch = ReplaySchedule(REPLAY_TOKENS, total); state = {"refresh": []}
    if method in STATIC:
        cyc = Cycler(SEL[(method, seed)], seed)
        return (lambda step, model: sch.take(step, cyc)), sch, state
    def fn(step, model):                                                     # E4 OPR-adapted
        if step % OPR_EVERY == 0:
            d = decision_slot(model, POOL.prompt.tolist(), DEV)
            agree = POOL[(d.score.values > 0).astype(int) == POOL.y.values]
            sel = select_by_class(agree, priority(agree, "random", seed=seed * 100003 + step), BUD)
            state["cyc"] = Cycler(sel, seed + step)
            state["refresh"].append({"step": step, "agree": float(len(agree) / len(POOL)), "pool_auroc": auroc(POOL.y, d.score),
                                     "selected": len(sel), "answer": int((sel.y == 0).sum()), "call": int((sel.y == 1).sum())})
        return sch.take(step, state["cyc"])
    return fn, sch, state

T_START = time.time()
RUNS = {}
def run(name, method, seed, beta):
    """Train one arm (or reuse an attached / existing finished run). Returns the final.pt path or None (time limit)."""
    batches = instruction_batches(SFT_TRAIN, CFG["batch_size"], seed); total = len(batches)
    assert total == REPORT07["sft_data"]["train_examples"] // CFG["batch_size"] + (REPORT07["sft_data"]["train_examples"] % CFG["batch_size"] > 0)
    out = f"{OUT}/{name}"
    for cand in [out] + [os.path.dirname(p) for p in sorted(glob.glob(f"/kaggle/input/**/08_replay/{name}/summary.json", recursive=True))]:
        sp = os.path.join(cand, "summary.json")
        if os.path.exists(sp) and os.path.exists(os.path.join(cand, "final.pt")):
            s = json.load(open(sp))
            if s["updates"] == total and abs(s["beta"] - beta) < 1e-12 and s.get("method") == method:
                RUNS[name] = s; stage(f"{name}: reusing finished run {cand}"); return os.path.join(cand, "final.pt")
    if (time.time() - T_START) / 3600 > MAX_HOURS:
        stage(f"{name}: SKIPPED (time limit {MAX_HOURS} h) - save this version, attach its output and re-run"); return None
    fn, sch, state = make_replay(method, seed, total)
    stage(f"{name}: method {method} | seed {seed} | beta {beta} | {total} updates")
    m = load_model(ROUTER, DEV)
    s = train_sft(m, batches, dict(CFG, log_every_steps=250), seed, out, replay=fn, beta=beta, log=lambda msg: stage(f"  [{name}] {msg.strip()}"))
    s.update(method=method, replay=sch.report(), opr_refresh=state["refresh"])
    json.dump(s, open(f"{out}/summary.json", "w"), indent=1, default=float)
    RUNS[name] = s; del m
    if DEV == "cuda": torch.cuda.empty_cache()
    stage(f"{name}: done in {s['minutes']} min | replay {s['replay']}")
    return f"{out}/final.pt"

## 5 · β on validation with the random arm (E2, seed 11)

In [ ]:
RP = [router_prompt(tok, q, "action") for q in V.question]
def val_auroc(path):
    m = load_model(path, DEV); d = decision_slot(m, RP, DEV); vl = sft_val_loss(m, SFT_VALX, DEV); del m
    return auroc(V.y, d.score), vl
BETA_ROWS = []
for beta in BETA_GRID:
    p = run(f"beta/E2_random_s{SEEDS[0]}_b{beta:g}", "random", SEEDS[0], beta)
    if p is None: break
    a, vl = val_auroc(p)
    BETA_ROWS.append({"beta": beta, "val_auroc": a, "instr_val_loss": vl, "path": p})
    stage(f"beta {beta:g}: val AUROC {a:.4f} | instruction val loss {vl:.4f}")
BETA_TABLE = pd.DataFrame(BETA_ROWS); display(BETA_TABLE.drop(columns="path").round(4))
assert len(BETA_TABLE) == len(BETA_GRID), "beta tuning incomplete (time limit) - re-run with this output attached"
best = BETA_TABLE.val_auroc.max()
BETA = float(BETA_TABLE[BETA_TABLE.val_auroc >= best - BETA_TOL].beta.min())
p07 = pd.read_csv(os.path.join(D07, "pilot_table.csv"))
e1a = p07[(p07.seed.astype(str) == str(SEEDS[0])) & (p07.frac == 1.0)].auroc
print(f"reference: E0 val AUROC {p07[p07.seed.astype(str) == 'all'].auroc.item():.4f} | E1 (no replay) seed {SEEDS[0]} {e1a.item():.4f}")
print(f"chosen beta = {BETA} (smallest within {BETA_TOL} AUROC of the best, {best:.4f})")

## 6 · Train every arm (E2–E7 × seeds); E2 seed 11 is the β-tuning run at the chosen β

In [ ]:
CKPTS = [("E0", "all", ROUTER)] + [("E1", s, E1[s]) for s in SEEDS]
for seed in SEEDS:
    for arm, meth in ARMS.items():
        if arm == "E2" and seed == SEEDS[0]:
            p = BETA_TABLE.set_index("beta").loc[BETA, "path"]
        else:
            p = run(f"{arm}_{meth}_s{seed}", meth, seed, BETA)
        if p is not None: CKPTS.append((arm, seed, p))
MISSING = [(a, s) for s in SEEDS for a in ARMS if (a, s) not in {(c[0], c[1]) for c in CKPTS}]
print(pd.DataFrame(CKPTS, columns=["arm", "seed", "path"]).to_string(index=False))
if MISSING: print("NOT TRAINED (time limit):", MISSING, "-> save version, attach this output, re-run")

## 7 · Evaluate every checkpoint on validation

In [ ]:
CP = [Q.closed_prompt_ids(tok, q) for q in V.question]
EP = []
for r in V.itertuples(index=False):
    sel, _ = select_non_overlapping(TOPDOCS[r.id], WART, WSTART, k=3, window_words=60)
    EP.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])

def evaluate(path):
    m = load_model(path, DEV)
    d = decision_slot(m, RP, DEV)
    gA = greedy_batch(m, CP, DEV, MAX_NEW, stop_ids=stop_ids); gC = greedy_batch(m, EP, DEV, MAX_NEW, stop_ids=stop_ids)
    d["pred_A"] = [PR.clean_generation(tok.decode(g)) for g in gA]; d["pred_C"] = [PR.clean_generation(tok.decode(g)) for g in gC]
    raw_A = [tok.decode(g) for g in gA]; raw_C = [tok.decode(g) for g in gC]
    d["fresh_em_A"] = [alias_em(p, g) for p, g in zip(d.pred_A, V.gold_aliases)]
    d["fresh_em_C"] = [alias_em(p, g) for p, g in zip(d.pred_C, V.gold_aliases)]
    d["contain_A"] = [alias_contains(p, g) for p, g in zip(raw_A, V.gold_aliases)]      # secondary: answer inside a sentence
    d["contain_C"] = [alias_contains(p, g) for p, g in zip(raw_C, V.gold_aliases)]
    vl = sft_val_loss(m, SFT_VALX, DEV); del m
    if DEV == "cuda": torch.cuda.empty_cache()
    for c in ["id", "group_id", "prop", "pop_bin", "dose_group", "y", "em_A", "em_C"]: d[c] = V[c].values
    return d, vl

PRED, VLOSS = {}, {}
os.makedirs(f"{OUT}/predictions", exist_ok=True)
for arm, seed, path in CKPTS:
    PRED[(arm, seed)], VLOSS[(arm, seed)] = evaluate(path)
    PRED[(arm, seed)].to_parquet(f"{OUT}/predictions/{arm}_s{seed}.parquet", index=False)
    p = PRED[(arm, seed)]
    stage(f"{arm} s{seed}: AUROC {auroc(p.y, p.score):.3f} | default call {np.mean(p.score > 0):.1%} | closed EM {p.fresh_em_A.mean():.1%} "
          f"| evidence EM {p.fresh_em_C.mean():.1%} | instr val loss {VLOSS[(arm, seed)]:.3f}")
P0 = PRED[("E0", "all")]
THR10 = float(np.quantile(P0.score, 0.9)); D0, D0T = P0.score.values > 0, P0.score.values >= THR10

## 8 · Results table (validation)

In [ ]:
def summarize(key):
    p = PRED[key]; s = p.score.values; d, dt = s > 0, s >= THR10
    ci = clustered_bootstrap_auroc(p.y, s, p.group_id.values, n_boot=N_BOOT)
    row = {"arm": key[0], "seed": key[1], "auroc": ci["auroc"], "auroc_lo": ci["lo"], "auroc_hi": ci["hi"], "auprc": auprc(p.y, s),
           "agree_E0_default": float((d == D0).mean()), "agree_E0_thr10": float((dt == D0T).mean()),
           "answer_to_call": int((~D0 & d).sum()), "call_to_answer": int((D0 & ~d).sum()), "default_call_rate": float(d.mean()),
           "instr_val_loss": VLOSS[key], "closed_em": p.fresh_em_A.mean(), "evidence_em": p.fresh_em_C.mean(),
           "closed_contain": p.contain_A.mean(), "evidence_contain": p.contain_C.mean(), "emission_rate": float(p.argmax_is_action.mean())}
    for b in BUDGETS:
        call = top_fraction(s, b); k = int(b * 100)
        row[f"U{k}_theta0"] = outcome(call, p.em_A, p.em_C)["accuracy"] - LAM * b                  # A3a
        row[f"U{k}_fresh_em"] = outcome(call, p.fresh_em_A, p.fresh_em_C)["accuracy"] - LAM * b     # A3b
        row[f"U{k}_fresh_contain"] = outcome(call, p.contain_A, p.contain_C)["accuracy"] - LAM * b
    return row
TABLE = pd.DataFrame([summarize(k) for k in PRED]); TABLE.to_csv(f"{OUT}/results_val.csv", index=False)
pd.set_option("display.width", 250)
display(TABLE[["arm", "seed", "auroc", "auroc_lo", "auroc_hi", "agree_E0_default", "agree_E0_thr10", "answer_to_call", "call_to_answer",
               "default_call_rate", "U10_theta0", "instr_val_loss"]].round(3))
display(TABLE[["arm", "seed", "closed_em", "evidence_em", "closed_contain", "evidence_contain", "U10_fresh_em", "U10_fresh_contain", "emission_rate"]].round(3))
AGG = TABLE[TABLE.arm != "E0"].groupby("arm")[["auroc", "agree_E0_default", "default_call_rate", "U10_theta0", "instr_val_loss"]].mean()
print("mean over seeds:"); display(AGG.round(3))

## 9 · Paired comparisons on validation (preliminary — claims only on test, notebook 09)

In [ ]:
def cmp(a, b, seed, metric):
    A, Bk = PRED[(a, seed)], PRED[(b, seed)]
    def f(idx, w):
        p = A if w == "a" else Bk
        if metric == "auroc": return auroc(p.y.values[idx], p.score.values[idx])
        return float(((p.score.values[idx] > 0) == D0[idx]).mean())                               # agreement with E0
    return paired_cluster_bootstrap(f, V.group_id.values, n_boot=N_BOOT, seed=seed)
CMP = []
for seed in SEEDS:
    for other in ["E1", "E2", "E3", "E4", "E5", "E6"]:
        if ("E7", seed) not in PRED or (other, seed) not in PRED: continue
        for metric in ["auroc", "agree_E0"]:
            r = cmp("E7", other, seed, metric); CMP.append({"seed": seed, "comparison": f"E7 UABR - {other}", "metric": metric, **r})
    for arm in ["E2", "E3", "E4", "E5", "E6"]:
        if (arm, seed) in PRED:
            r = cmp(arm, "E1", seed, "auroc"); CMP.append({"seed": seed, "comparison": f"{arm} - E1", "metric": "auroc", **r})
CMP = pd.DataFrame(CMP); CMP["excludes_0"] = (CMP.ci_lo > 0) | (CMP.ci_hi < 0)
CMP.to_csv(f"{OUT}/comparisons_val.csv", index=False)
display(CMP.drop(columns="n_boot_valid").round(4))

## 10 · Mechanism: retention by pre-SFT margin, and OPR buffer

In [ ]:
qb = pd.qcut(P0.margin, 5, labels=False, duplicates="drop") + 1
rows = []
for (arm, seed), p in PRED.items():
    if arm == "E0": continue
    keep = (p.score.values > 0) == D0
    for b in sorted(qb.unique()):
        rows.append({"arm": arm, "seed": seed, "margin_quintile": int(b), "retention": float(keep[qb == b].mean())})
RET = pd.DataFrame(rows).pivot_table(index=["arm", "seed"], columns="margin_quintile", values="retention")
print("share of pre-SFT decisions kept, by pre-SFT margin quintile (1 = smallest margin):"); display(RET.round(3))
for name, s in RUNS.items():
    if s.get("opr_refresh"):
        print(name, "OPR refreshes:"); display(pd.DataFrame(s["opr_refresh"]).round(3))

## 11 · Save + validity checks

In [ ]:
meta = {"notebook": "08_replay", "beta": BETA, "beta_table": BETA_TABLE.drop(columns="path").to_dict("records"), "tau": TAU,
        "pool": {"prompts": len(POOL), "tokens": POOL_TOKENS, "call_share": float(POOL.y.mean())}, "select_tokens": SELECT_TOKENS,
        "class_budgets": BUD, "replay_tokens": REPLAY_TOKENS, "instruction_tokens": INSTR_TOKENS, "lambda": LAM, "seeds": SEEDS,
        "runs": RUNS, "selection_report": SEL_REPORT.to_dict("records"), "jaccard": JAC.to_dict(), "missing": MISSING,
        "checkpoints": [[a, str(s), p] for a, s, p in CKPTS], "router": ROUTER, "e1_from": D07,
        "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}
json.dump(meta, open(f"{OUT}/meta.json", "w"), indent=1, default=float)
for meth in STATIC:
    for seed in SEEDS: SEL[(meth, seed)].drop(columns=["prompt"]).to_parquet(f"{OUT}/selection_{meth}_s{seed}.parquet", index=False)
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("all_arms_trained", not MISSING, f"missing {MISSING}" if MISSING else "6 arms x %d seeds" % len(SEEDS))
ups = {s["updates"] for s in RUNS.values()}
add("equal_updates", len(ups) == 1, f"updates {ups}")
rt = [s["replay"]["replay_tokens"] for s in RUNS.values()]
add("equal_replay_tokens", max(rt) - min(rt) <= max(0.01 * min(rt), int(POOL.tokens.max())), f"replay tokens {min(rt):,}-{max(rt):,} (target {REPLAY_TOKENS:,}; tolerance 1 % or one example)")
share = {n: s["replay"]["targets_call"] / max(1, s["replay"]["replay_rows"]) for n, s in RUNS.items()}
static_share = [v for n, v in share.items() if "opr" not in n]
add("class_ratio_equal_static_arms", max(static_share) - min(static_share) < 0.03, f"CALL share of replay rows {min(static_share):.3f}-{max(static_share):.3f} (pool {POOL.y.mean():.3f})")
opr_share = [v for n, v in share.items() if "opr" in n]
add("class_ratio_opr", all(abs(v - np.mean(static_share)) < 0.05 for v in opr_share), f"OPR CALL share {[round(v, 3) for v in opr_share]}", "warn")
e1 = {s: TABLE[(TABLE.arm == "E1") & (TABLE.seed == s)].auroc.item() for s in SEEDS}
p07 = pd.read_csv(os.path.join(D07, "pilot_table.csv")); p07 = p07[p07.frac == 1.0]
ref = {int(r.seed): r.auroc for r in p07.itertuples()}
add("E1_reproduces_07", all(abs(e1[s] - ref[s]) < 2e-3 for s in SEEDS if s in ref), f"E1 AUROC {e1} vs 07 {ref}")
e0 = TABLE[TABLE.arm == "E0"].auroc.item()
d0max = float(np.abs(P0.score.values - V.router_score.values).max())
add("E0_is_06_router", d0max < 1e-2, f"E0 AUROC {e0:.4f} | max |score - 06 router_score| {d0max:.1e}")
vl1 = np.mean([VLOSS[("E1", s)] for s in SEEDS]); worst = max(v for k, v in VLOSS.items() if k[0] not in ("E0", "E1"))
add("replay_keeps_SFT_learning", worst <= vl1 + 0.05, f"instruction val loss E1 {vl1:.3f}, worst replay arm {worst:.3f}", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "phase6_validation", "status": status, "beta": BETA, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
for f in sorted(glob.glob(f"{OUT}/*")):
    if os.path.isfile(f): print(f"{os.path.getsize(f)/1e6:9.1f} MB  {os.path.basename(f)}")
print(f"\nPHASE 6 (validation): {status} | beta {BETA}")

## Next
Notebook 09 opens the sealed **test** split once: E0, E1 and E2–E7 (both seeds) with the claim rule fixed in advance
(UABR better than X only if the 95 % CI excludes 0 in favour of UABR in **both** seeds on the primary endpoint).